# 外部text manifestでT2TTを学習・評価する

JParaCrawl等で**外部生成済みのJA–EN JSONL**を使います。必要な列は `pair_id / split / ja_text / en_text`。train/dev必須、testと音声は不要です。このNotebookはデータのfiltering・TTS・Whisper QC・S2UT学習を実行しません。

**順序**：Drive接続 → 設定 → 環境 → textをVMへstaging → 学習/復元 → dev BLEU・翻訳例 → 任意のAttention。CheckpointはVMに書き、固定benchmarkの検証付きZIPでDriveへ保存します。切断後も同じrevision・設定・保存先で上から再実行します。


In [ ]:
#@title 共通ログ：開始・逐次出力・生存通知・完了・失敗
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/jparacrawl_t2tt/bootstrap')


In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


## 1. 入力と独立した保存先

`SOURCE_MANIFEST`を外部JSONLのDrive上のパスへ変更します。原本は読み取り専用です。`RUN_ID`は新しい実験名にしてください。tokenizerを変える場合は別stageへ分離します。`RUN_TRAINING=False`は完了済み学習の復元・評価用です。


In [ ]:
#@title 入力・実験・学習設定
from pathlib import Path
import os, sys, json
SOURCE_MANIFEST = '/content/drive/MyDrive/SET_TEXT_MANIFEST.jsonl' #@param {type:"string"}
RUN_ID = 'jparacrawl-t2tt-001' #@param {type:"string"}
TOKENIZER = 'sentencepiece' #@param ['character', 'sentencepiece']
MAX_UPDATES = 200000 #@param {type:"integer"}
RUN_TRAINING = True #@param {type:"boolean"}
RUN_ATTENTION = False #@param {type:"boolean"}
ALL_LAYERS = True #@param {type:"boolean"}
SNAPSHOT_UPDATES = 10000 #@param {type:"integer"}
KEEP_LAST = 2 #@param {type:"integer"}
ROOT = Path('/content/alignment-transfer-work') / RUN_ID
PERSISTENT = Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / RUN_ID
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '79860e497efc16e280b5063a5de36533a007a912'
REPO = Path('/content/alignment-transfer-s2ut')
PYTHON = '/content/alignment-venv/bin/python'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
BENCHMARK_RUNTIME_REVISION = '4361e55a99c7f4ea3a00c5c138f6820c34a07d06'
os.environ['MPLBACKEND'] = 'Agg'
if not RUN_ID or Path(RUN_ID).name != RUN_ID or RUN_ID in ('.','..'):
    raise ValueError('RUN_ID must be a single new directory name')
if TOKENIZER not in ('character','sentencepiece') or MAX_UPDATES < 1:
    raise ValueError('invalid tokenizer or update budget')
if 'SET_' in SOURCE_MANIFEST:
    raise ValueError('SOURCE_MANIFESTを実際のtext JSONLへ変更してください')
LOGS = ROOT / 'execution'
MANIFEST = ROOT / 'text-manifest/manifest.jsonl'
CONFIG = REPO / 'configs/t2tt/fisher.yaml'
TRAIN_STAGE = 't2tt-' + TOKENIZER
WORK = ROOT / TRAIN_STAGE
BACKUP = PERSISTENT / 'stages' / TRAIN_STAGE
CHECKPOINT = WORK / 'checkpoint_best.pt'
EVAL_ROOT = ROOT / ('dev-' + TOKENIZER)
EVAL_BACKUP = PERSISTENT / 'stages' / ('dev-' + TOKENIZER)
DEV_METRICS = EVAL_ROOT / 'dev-text.json'
ATTENTION = ROOT / ('attention-' + TOKENIZER + ('-all-layers' if ALL_LAYERS else ''))
ATTENTION_BACKUP = PERSISTENT / 'stages' / ATTENTION.name
print('VM:', ROOT, '\nDrive:', PERSISTENT, '\nmanifest:', MANIFEST)


## 2. 環境を構築・検査

既存Notebookと同じPython 3.10.22・固定fairseq/benchmark runtimeを導入します。音声モデルの取得はしません。最初の構築には時間がかかります。すべてのコマンドで開始・PID・逐次ログ・10秒の生存通知・終了を表示/保存します。


In [ ]:
#@title Python・fairseq環境を構築
if not (REPO / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', REPO_URL, str(REPO), name='clone-alignment')
execute('git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION, name='fetch-alignment')
execute('git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION, name='checkout-alignment')
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)


In [ ]:
#@title benchmark runtimeを固定revisionで導入
if not (BENCHMARK_RUNTIME / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', str(BENCHMARK_RUNTIME))
execute('git', '-C', str(BENCHMARK_RUNTIME), 'fetch', 'origin', BENCHMARK_RUNTIME_REVISION, name='fetch-benchmark-runtime')
execute('git', '-C', str(BENCHMARK_RUNTIME), 'checkout', '--detach', BENCHMARK_RUNTIME_REVISION, name='checkout-benchmark-runtime')
execute(PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME))
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
execute(PYTHON, '-m', 'alignment_transfer_s2ut.environment', '--native', '--benchmark', '--cuda', name='environment')
execute(PYTHON, '-m', 'pytest', 'tests/test_text_manifest.py', 'tests/test_text_t2tt.py', '-q', '--basetemp', '/content/text-t2tt-tests', name='text-cpu-preflight')


## 3. TextだけをVMへコピー

原本は一度だけ読み、4列を順序・原文そのままでローカルへ保存します。音声パス・SHA・その他の列はT2TT入力から除きます。文を削除・追加・正規化したりsplitを作り直したりしません。

後から原本に音声列を追加しても、4列が同じならローカルmanifestと学習identityは変わりません。実際のtext/splitが変わった場合は新しいRUN_IDを使います。


In [ ]:
prepared = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_manifest', '--input', SOURCE_MANIFEST, '--output', str(MANIFEST), '--resume', name='prepare-external-text')
print(json.dumps(prepared, ensure_ascii=False, indent=2))


## 4. T2TT学習・復元

trainだけで語彙を作り、dev lossでbestを選びます。SentencePieceは日英別Unigram・各8000目標・identity。characterも選択できます。既存character/SPM runをseedしません。

初回は0から、次回はこのstage自身のlast・optimizer・RNG・cursorから再開します。既定は200k、batch 8×4、max tokens 20000、validation 1000、VM save 100。Driveは10kごと＋milestone＋終了時、最新2＋milestoneを保持します。`[T2TT loss]`はローカル実測、`durable_updates`は保存済み更新です。

学習中に別セルを実行しても終了まで待機します。履歴を読むために重いZIPを毎回取得する必要はなく、各snapshotの`metrics-summary.json`を利用できます。


In [ ]:
if RUN_TRAINING:
    training_args = ['--manifest', str(MANIFEST), '--config', str(CONFIG), '--output', str(WORK),
        '--backend', 'fairseq', '--device', 'cuda', '--tokenizer', TOKENIZER, '--max-updates', str(MAX_UPDATES),
        '--batch-size', '8', '--update-frequency', '4', '--max-tokens', '20000', '--loader-workers', '2',
        '--validation-interval', '1000', '--save-interval-updates', '100', '--resume-if-exists']
    if TOKENIZER == 'sentencepiece':
        training_args += ['--source-vocab-size', '8000', '--target-vocab-size', '8000', '--sentencepiece-model-type', 'unigram']
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', TRAIN_STAGE, '--logs', str(LOGS),
        '--monitor-gpu', '--', PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'train',
        '--work', str(WORK), '--backup', str(BACKUP), '--snapshot-updates', str(SNAPSHOT_UPDATES), '--keep-last', str(KEEP_LAST),
        '--', PYTHON, '-m', 'alignment_transfer_s2ut.train_t2tt', *training_args, name='text-t2tt-training')
else:
    log_event(TRAIN_STAGE, 'skipped', LOGS, 'training disabled; restore completed checkpoint below')


## 5. 同じdevでBLEUと翻訳例を確認

新VMでは検証済みZIPから復元します。完了済みMAX_UPDATESと入力text hashを確認し、bestを評価します。既定beam 5・length penalty 1・同じSacreBLEUを使用し、参照訳を生成器へ渡しません。結果・prediction JSONL・cacheも別の検証付きstageへ保存します。


In [ ]:
restored = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'restore-inputs',
    '--work', str(WORK), '--backup', str(BACKUP), '--expected-updates', str(MAX_UPDATES), name='restore-text-t2tt')
status = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'status', '--work', str(WORK),
    '--target-updates', str(MAX_UPDATES), name='verify-text-t2tt-complete')
print(status)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(EVAL_ROOT), '--backup', str(EVAL_BACKUP), '--',
    PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', str(MANIFEST), '--checkpoint', str(CHECKPOINT),
    '--split', 'dev', '--beam-size', '5', '--length-penalty', '1', '--device', 'cuda', '--output', str(DEV_METRICS), '--resume', name='text-dev-bleu')
metrics = json.loads(DEV_METRICS.read_text(encoding='utf-8'))
print(json.dumps({k:v for k,v in metrics.items() if k not in ('evaluation_contract',)}, ensure_ascii=False, indent=2))


In [ ]:
import pandas as pd
from IPython.display import display, Image
predictions = [json.loads(line) for line in DEV_METRICS.with_name(DEV_METRICS.stem + '-predictions.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
references = {row['pair_id']:row['en_text'] for row in (json.loads(line) for line in MANIFEST.read_text(encoding='utf-8').splitlines() if line.strip()) if row['split']=='dev'}
display(pd.DataFrame([{'pair_id':r['pair_id'], 'JA':r['source_text'], 'Reference EN':references[r['pair_id']],
    'T2TT EN':r['hypothesis'], 'EOS':r['eos_reached']} for r in predictions[:10]]))
diagnostic = WORK / 'tokenization.json'
if diagnostic.is_file():
    tokenization = json.loads(diagnostic.read_text(encoding='utf-8'))
    print('actual vocab sizes:', tokenization['source']['actual_vocab_size'], tokenization['target']['actual_vocab_size'])
    display(pd.DataFrame(tokenization['statistics']).T)
execute(PYTHON, '-m', 'alignment_transfer_s2ut.text_t2tt_runtime', 'report', '--work', str(WORK),
    '--output', str(WORK / 'diagnostics'), name='text-learning-curve')
display(Image(filename=str(WORK / 'diagnostics/learning-curve.png')))


## 6. 任意：Text Attentionを抽出

品質を確認してから設定の`RUN_ATTENTION=True`にします。音声不要です。全層なら対応層を1 forwardで取得し、subwordの原文codepoint spanも保存します。後から同じID/split/textに音声を追加すれば、CTC/unitを別途用意してteacherへ投影できます。ここでは音声生成・CTC・S2UTは実行しません。


In [ ]:
if RUN_ATTENTION:
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(ATTENTION), '--backup', str(ATTENTION_BACKUP), '--',
        PYTHON, '-m', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', str(MANIFEST), '--checkpoint', str(CHECKPOINT),
        '--output', str(ATTENTION), '--device', 'cuda', '--resume', *(['--all-layers'] if ALL_LAYERS else []), name='text-attention')
else:
    log_event('text-attention', 'skipped', LOGS, 'set RUN_ATTENTION=True after reviewing dev translations')


## 保存先と再開

- 元manifest：`SOURCE_MANIFEST`（read-only）
- canonical text：`ROOT/text-manifest/manifest.jsonl`
- last/best/model/metrics：`ROOT/t2tt-<tokenizer>/`
- checkpoint ZIP：`PERSISTENT/stages/t2tt-<tokenizer>/`
- dev/Attention ZIP：それぞれの独立stage

ランタイム切断後は同じ設定・固定revisionで上から実行します。元textが変わっていなければcanonical bytesは変わりません。最後の検証済みsnapshot以降の更新は再計算されます。別コーパス・split・学習条件には新しいRUN_IDを使ってください。[API・identityと互換性](../../docs/TEXT_ONLY_T2TT.md)
